## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key not set
Google API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)
Groq API Key exists and begins gsk_


In [4]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [7]:
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

STEP 3: Create a model object

In [11]:
groq_model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=groq_client,
)

In [14]:
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=groq_client,
)

print(model)

In [15]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent", instructions=instructions, model=model)

In [16]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [17]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [18]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [19]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [20]:
tools = [tool1, tool2, tool3, send_email_tool]

In [24]:
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=groq_client,
)

In [25]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=model)

In [29]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


The best cold sales email was sent to the user. The email aims to simplify the SOC2 compliance journey with ComplAI, an innovative SaaS tool that automates compliance checks, streamlines audit preparation, enhances collaboration, and demonstrates compliance. The email highlights the key benefits of using ComplAI, including reducing compliance costs by up to 70%, cutting audit preparation time by up to 50%, and improving audit success rates by up to 90%. The email also includes a testimonial from a satisfied customer and invites the user to schedule a demo to learn more about ComplAI.


OPENAI_API_KEY is not set, skipping trace export


## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [30]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [31]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [32]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Nawanshu 
"""

In [38]:
checker = Agent(
    name="Checker",
    instructions="You review potential sales emails",
    model="gpt-5.4-mini",
    output_type=EmailReview,
)

In [42]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model=model, output_type=EmailReview)
result = await Runner.run(checker, email)

OPENAI_API_KEY is not set, skipping trace export


In [43]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=4, contains_placeholders=True)

In [44]:
review.is_professional

False

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [49]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [50]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=model, output_guardrails=[email_guardrail])

In [53]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "sales_writer.run", "arguments": {\n  "request": "Write a cold sales email for ComplAI, a conversational AI platform that helps businesses automate customer support and improve engagement. The tone should be like a cowboy, with a friendly, rugged style. Include a compelling subject line, brief intro, value proposition, social proof, and a clear call to action."\n}}'}}

OPENAI_API_KEY is not set, skipping trace export


Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [54]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


**Subject:** 🤠 How ComplAI Can Keep Your Business Riding Straight & True  

---

Howdy **[First Name]**,

I’m **[Your Name]**, the sales wrangler over at **ComplAI**, and I reckon you’ve been wrangling a heap of compliance paperwork lately. Let me cut straight to the chase—our AI‑powered platform can saddle up that mountain of rules, regulations, and reporting so you can spend more time on the trail ahead and less stuck in the mud.

**Why ComplAI is the trusty side‑kick you need:**

- **Rodeo‑Fast Audits:** AI that spots gaps faster than a stallion on open range.  
- **Zero‑Tolerant Errors:** Automated checks keep you compliant, so regulators don’t come a‑knocking.  
- **Money‑Saving Trailblaze:** Cut compliance costs by up to 40%, freeing cash for the real gold‑rush projects.  
- **Easy‑Peasy Integration:** Plug‑and‑play with your existing tools—no need to reinvent the wagon.

Folks in **[Industry/Company Size]** have already hit the trail with us and are seeing smoother rides and few

OPENAI_API_KEY is not set, skipping trace export


In [55]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

OPENAI_API_KEY is not set, skipping trace export


The email is not professional or has placeholders and will not be sent


OPENAI_API_KEY is not set, skipping trace export


## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [56]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [57]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [58]:
CODE_DIR

PosixPath('/Users/nick/projects/agents2/2_openai/code')

In [59]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [60]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [61]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [62]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model=model, default_manifest=manifest, capabilities=capabilities)

In [69]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

Sure thing! Could you please paste the code that's giving you trouble (or describe the unexpected behavior you’re seeing)? That’ll help me pinpoint the bug and suggest a fix.


OPENAI_API_KEY is not set, skipping trace export


## OPTIONAL EXTRA: MCP Teaser!

In [65]:
from agents.mcp import MCPServerStreamableHttp


In [66]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [68]:
agent = Agent(name="Expert", instructions="Answer the question", model=model)
result = await Runner.run(agent, task)
print(result.final_output)

I’m not aware of the specifics of the “SandboxAgents” feature or the role of a Manifest object in the OpenAI Agents SDK as of May 2026. My training data only goes up through mid‑2024, and I don’t have information on features that were released after that point. If you have documentation or details you can share, I’d be happy to help interpret them!


OPENAI_API_KEY is not set, skipping trace export


In [70]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model=model)
    result = await Runner.run(agent, task)

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


The **`Manifest`** is the sandbox’s “blue‑print”.  
When you create a `SandboxAgent` you pass a `default_manifest` that:

* **Declares the sandbox users** (e.g., the `User` that the agent will run as).  
* **Lists the filesystem entries** (directories, files, etc.) that will be mounted into the sandbox.  
* **Specifies the permissions** for each entry (read, write, execute, and which user/group can use them).

In short, the Manifest defines the initial workspace layout and the access‑control rules for that workspace, allowing the agent to know what files/directories it can read, write, or execute and under which user identity those permissions apply.


And see the traces:

https://platform.openai.com/traces
